In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower, regexp_replace, substring, length

spark = (
    SparkSession.builder
    .appName("capacity-simulation")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/25 10:52:57 WARN Utils: Your hostname, CrisBook.local, resolves to a loopback address: 127.0.0.1; using 192.168.13.159 instead (on interface en0)
26/04/25 10:52:57 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/25 10:52:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/25 10:52:58 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
df = spark.read.parquet("../data/dev/gold/vulnerabilities_clustered")

df.printSchema()
df.show(5, truncate=False)

root
 |-- cve_id: string (nullable = true)
 |-- description: string (nullable = true)
 |-- cwe: string (nullable = true)
 |-- published: string (nullable = true)
 |-- lastModified: string (nullable = true)
 |-- cvss_score: double (nullable = true)
 |-- cvss_severity: string (nullable = true)
 |-- kev_date_added: date (nullable = true)
 |-- required_action: string (nullable = true)
 |-- known_ransomware_campaign_use: string (nullable = true)
 |-- epss_score: double (nullable = true)
 |-- epss_percentile: double (nullable = true)
 |-- is_kev: integer (nullable = true)
 |-- cvss_normalized: double (nullable = true)
 |-- priority_score: double (nullable = true)
 |-- priority_level: string (nullable = true)
 |-- cluster_id: integer (nullable = true)

+--------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [3]:
from pyspark.sql.functions import col, coalesce, lit
from pyspark.sql.types import DoubleType, IntegerType

df = (
    df
    .withColumn("priority_score", coalesce(col("priority_score").cast(DoubleType()), lit(0.0)))
    .withColumn("cvss_score", coalesce(col("cvss_score").cast(DoubleType()), lit(0.0)))
    .withColumn("epss_score", coalesce(col("epss_score").cast(DoubleType()), lit(0.0)))
    .withColumn("is_kev", coalesce(col("is_kev").cast(IntegerType()), lit(0)))
    .withColumn("cluster_id", coalesce(col("cluster_id").cast(IntegerType()), lit(-1)))
)

In [4]:
PATCH_CAPACITY = 50

In [5]:
from pyspark.sql.functions import rand

random_strategy = (
    df
    .orderBy(rand(seed=42))
    .limit(PATCH_CAPACITY)
)

In [6]:
cvss_strategy = (
    df
    .orderBy(col("cvss_score").desc())
    .limit(PATCH_CAPACITY)
)

In [7]:
priority_strategy = (
    df
    .orderBy(col("priority_score").desc())
    .limit(PATCH_CAPACITY)
)

In [8]:
hybrid_strategy = (
    df
    .orderBy(
        col("is_kev").desc(),
        col("priority_score").desc()
    )
    .limit(PATCH_CAPACITY)
)

In [9]:
from pyspark.sql.functions import avg, sum as spark_sum, countDistinct, count, round

def evaluate_strategy(strategy_df, strategy_name):
    result = (
        strategy_df
        .agg(
            count("*").alias("selected_cves"),
            round(avg("priority_score"), 4).alias("avg_priority_score"),
            round(avg("cvss_score"), 4).alias("avg_cvss_score"),
            round(avg("epss_score"), 4).alias("avg_epss_score"),
            spark_sum("is_kev").alias("kev_selected"),
            countDistinct("cluster_id").alias("clusters_covered")
        )
        .withColumn("strategy", lit(strategy_name))
    )
    
    return result.select(
        "strategy",
        "selected_cves",
        "avg_priority_score",
        "avg_cvss_score",
        "avg_epss_score",
        "kev_selected",
        "clusters_covered"
    )

In [10]:
random_eval = evaluate_strategy(random_strategy, "Random")
cvss_eval = evaluate_strategy(cvss_strategy, "CVSS-only")
priority_eval = evaluate_strategy(priority_strategy, "Priority score")
hybrid_eval = evaluate_strategy(hybrid_strategy, "KEV-first + priority")

comparison_df = (
    random_eval
    .union(cvss_eval)
    .union(priority_eval)
    .union(hybrid_eval)
)

comparison_df.show(truncate=False)

+--------------------+-------------+------------------+--------------+--------------+------------+----------------+
|strategy            |selected_cves|avg_priority_score|avg_cvss_score|avg_epss_score|kev_selected|clusters_covered|
+--------------------+-------------+------------------+--------------+--------------+------------+----------------+
|Random              |50           |0.2191            |5.448         |0.003         |0           |5               |
|CVSS-only           |50           |0.4909            |10.0          |0.1473        |8           |3               |
|Priority score      |50           |0.9672            |9.832         |0.9347        |50          |2               |
|KEV-first + priority|50           |0.9672            |9.832         |0.9347        |50          |2               |
+--------------------+-------------+------------------+--------------+--------------+------------+----------------+



In [11]:
hybrid_strategy.select(
    "cve_id",
    "priority_score",
    "cvss_score",
    "epss_score",
    "is_kev",
    "cluster_id",
    "description"
).show(20, truncate=False)

+--------------+--------------+----------+----------+------+----------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [12]:
hybrid_strategy.groupBy("cluster_id").count().orderBy(col("count").desc()).show()

+----------+-----+
|cluster_id|count|
+----------+-----+
|         6|   49|
|         3|    1|
+----------+-----+



In [13]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

PATCH_CAPACITY = 50
MAX_PER_CLUSTER = 10

w_cluster = Window.partitionBy("cluster_id").orderBy(col("priority_score").desc())

diverse_strategy = (
    df
    .withColumn("rank_in_cluster", row_number().over(w_cluster))
    .filter(col("rank_in_cluster") <= MAX_PER_CLUSTER)
    .orderBy(col("priority_score").desc())
    .limit(PATCH_CAPACITY)
)

In [14]:
diverse_strategy.groupBy("cluster_id").count().orderBy("cluster_id").show()

+----------+-----+
|cluster_id|count|
+----------+-----+
|         1|    8|
|         2|   10|
|         3|   10|
|         6|   10|
|         9|    8|
|        10|    4|
+----------+-----+



In [15]:
diverse_eval = evaluate_strategy(diverse_strategy, "Cluster-aware hybrid")

comparison_df = (
    random_eval
    .union(cvss_eval)
    .union(priority_eval)
    .union(hybrid_eval)
    .union(diverse_eval)
)

comparison_df.show(truncate=False)

+--------------------+-------------+------------------+--------------+--------------+------------+----------------+
|strategy            |selected_cves|avg_priority_score|avg_cvss_score|avg_epss_score|kev_selected|clusters_covered|
+--------------------+-------------+------------------+--------------+--------------+------------+----------------+
|Random              |50           |0.2191            |5.448         |0.003         |0           |5               |
|CVSS-only           |50           |0.4909            |10.0          |0.1473        |8           |3               |
|Priority score      |50           |0.9672            |9.832         |0.9347        |50          |2               |
|KEV-first + priority|50           |0.9672            |9.832         |0.9347        |50          |2               |
|Cluster-aware hybrid|50           |0.773             |9.172         |0.7352        |28          |6               |
+--------------------+-------------+------------------+--------------+--

In [16]:
cluster_risk_df = (
    df
    .groupBy("cluster_id")
    .agg(
        count("*").alias("total_cves"),
        round(avg("priority_score"), 4).alias("avg_priority_score"),
        spark_sum("is_kev").alias("total_kev")
    )
    .orderBy(col("avg_priority_score").desc())
)

cluster_risk_df.show(truncate=False)

+----------+----------+------------------+---------+
|cluster_id|total_cves|avg_priority_score|total_kev|
+----------+----------+------------------+---------+
|8         |83        |0.266             |0        |
|2         |7585      |0.2603            |0        |
|3         |1043      |0.2573            |16       |
|10        |1273      |0.2532            |17       |
|1         |697       |0.2428            |5        |
|5         |348       |0.2323            |0        |
|6         |79347     |0.2321            |335      |
|0         |4157      |0.2053            |4        |
|11        |48        |0.1993            |0        |
|7         |300       |0.1977            |0        |
|4         |34        |0.1938            |0        |
|9         |1064      |0.158             |2        |
+----------+----------+------------------+---------+



In [17]:
hybrid_strategy.write.mode("overwrite").parquet("../data/dev/gold/remediation_recommendations")
diverse_strategy.write.mode("overwrite").parquet("../data/dev/gold/cluster_aware_recommendations")
comparison_df.write.mode("overwrite").parquet("../data/dev/gold/strategy_comparison")
cluster_risk_df.write.mode("overwrite").parquet("../data/dev/gold/cluster_risk_summary")

26/04/25 13:30:09 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 306101 ms exceeds timeout 120000 ms
26/04/25 13:30:09 WARN SparkContext: Killing executors is not supported by current scheduler.
26/04/25 13:30:12 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$